#  Outlier Management : Detection & Treatment




In real-world data, some observations do not follow the same statistical pattern as the rest of the dataset. These are called **outliers**. Whether they come from measurement errors, data entry mistakes, or rare but genuine events, knowing how to detect and handle them is a fundamental step in any data science pipeline.

This notebook covers the full outlier management pipeline :
- **Detection**: identify which points are outliers and why
- **Treatment**: decide what to do with them once identified

---
## 1. Key Concepts

### What is an outlier?

An **outlier** is an observation that lies at an abnormal distance from other values in a dataset. It does not follow the same statistical pattern as the majority of observations.

Outliers can come from several sources:

- **Data entry errors** — a human typed 2000 instead of 200
- **Sensor failures** — a broken thermometer recording −999°C
- **Rare genuine events** — a fraudulent transaction, a record-breaking salary
- **Sampling issues** — a sample that came from a different population

This distinction matters: not every outlier should be removed. A fraud case *is* an outlier, and it is exactly the point you want to keep.

### Why do outliers matter?

Most machine learning algorithms are sensitive to extreme values. A single outlier can significantly shift the mean, inflate the standard deviation, distort regression lines, and bias clusters in k-means. The table below summarizes the main risks:

| Impact | Description |
|--------|-------------|
| **Model degradation** | Linear regression, k-means, PCA are all directly affected by extreme values |
| **Biased statistics** | The mean and std are both pulled toward extreme values |
| **Signal vs. noise** | Some outliers carry real information (fraud, rare disease) |
| **Data quality** | Outliers often reveal hidden problems in data collection |

### Univariate vs. Multivariate Outliers

There are two fundamentally different kinds of outliers:

```
UNIVARIATE   → A value that is extreme in a SINGLE feature
               Example: age = 200 in a dataset of humans
               → detectable by looking at one column at a time

MULTIVARIATE → A COMBINATION of values that is abnormal together,
               even if each value alone looks normal
               Example: height = 150 cm AND weight = 120 kg
               → only detectable by looking at all features simultaneously
```

This distinction drives the choice of both detection and treatment method.


---
## 2. Detection Methods

---

### 2.1 IQR — Interquartile Range

The IQR method is one of the oldest and most intuitive outlier detection techniques. It is based entirely on the distribution of the data itself, without any assumption about its shape.

**Core idea:** define a "normal zone" using the middle 50% of the data (between Q1 and Q3). Any point that falls too far outside this zone is flagged as an outlier.

The IQR is computed as the difference between the third and first quartiles:

$$IQR = Q_3 - Q_1$$

Two fences are then derived using a factor $k$ (default 1.5):

$$\text{Lower fence} = Q_1 - k \times IQR \qquad \text{Upper fence} = Q_3 + k \times IQR$$

A point $x$ is flagged as an outlier if:
$$x < \text{Lower fence} \quad \text{or} \quad x > \text{Upper fence}$$

With $k = 1.5$, approximately 99.3% of a normal distribution falls within the fences. Using $k = 3.0$ restricts detection to only extreme outliers.


---

### 2.2 Z-Score

The Z-Score method standardizes each value relative to the column mean and standard deviation. It expresses how many standard deviations a point is from the center of the distribution.

$$Z_i = \frac{x_i - \mu}{\sigma}$$

A point is flagged as an outlier if its Z-Score exceeds a threshold in absolute value:

$$|Z_i| > \text{threshold} \quad (\text{default: } 3.0)$$

Under a normal distribution, 99.7% of values fall within 3 standard deviations of the mean. Any point beyond that is considered statistically unusual.

 **Important limitation:** both the mean $\mu$ and the standard deviation $\sigma$ are themselves sensitive to outliers. If the data contains extreme values, they will inflate $\sigma$ and shift $\mu$, making the Z-Score less reliable. The IQR method does not have this problem.

---

### 2.3 Isolation Forest

Isolation Forest takes a completely different approach: instead of measuring how extreme a point is, it measures how **easy it is to isolate**.

**principle:** an outlier is a point that can be separated from the rest of the data with very few random cuts. A normal point, embedded in a dense cluster, requires many more cuts to be isolated.



The algorithm builds an ensemble of random isolation trees. For each tree:
1. Draw a random sub-sample of $\psi$ points from the data
2. Randomly select a feature and a random split value within its range
3. Recursively partition the data until each point is isolated

The **anomaly score** of a point $x$ is derived from its average path length $h(x)$ across all trees:

$$s(x, \psi) = 2^{-\frac{E[h(x)]}{c(\psi)}}$$

where $c(\psi)$ is a normalization constant (the expected path length of an unsuccessful search in a binary tree of size $\psi$).



| Score | Interpretation |
|-------|---------------|
| $s \approx 1$ | Very short path → isolated quickly → **outlier** |
| $s \approx 0.5$ | Average path → similar to normal points → **normal** |
| $s \approx 0$ | Very long path → deeply embedded → **clearly normal** |
---

### 2.4 Local Outlier Factor (LOF)

IQR and Z-Score use **global** statistics — they compare each point to the entire dataset. LOF takes a fundamentally different approach: it compares each point to its **local neighborhood**.

**Core idea:** a point is an outlier if it sits in a region that is much less dense than the regions its neighbors come from. A point that seems normal globally can be anomalous locally.



The algorithm computes four quantities in sequence:

**① k-distance of point $p$** — the distance to its $k$-th nearest neighbor:
$$k\text{-distance}(p) = d\left(p,\ k\text{-th nearest neighbor of } p\right)$$

**② Reachability distance** from $p$ to neighbor $o$ — smooths distances in dense areas to avoid instability:
$$\text{reach-dist}_k(p, o) = \max\left(k\text{-distance}(o),\ d(p, o)\right)$$

If $p$ is very close to $o$, we still assign it a minimum distance of $o$'s own $k$-distance. This prevents artificially small distances in dense regions.

**③ Local Reachability Density (lrd)** of $p$ — the inverse of the average reachability distance to its neighbors:
$$\text{lrd}_k(p) = \frac{1}{\dfrac{1}{|N_k(p)|}\sum_{o \in N_k(p)} \text{reach-dist}_k(p, o)}$$

A high lrd means $p$ is in a dense region. A low lrd means $p$ is in a sparse region.

**④ LOF Score** of $p$ — the ratio of the average lrd of $p$'s neighbors to the lrd of $p$ itself:
$$LOF_k(p) = \frac{\dfrac{1}{|N_k(p)|}\sum_{o \in N_k(p)} \text{lrd}_k(o)}{\text{lrd}_k(p)}$$



| LOF Score | Interpretation |
|-----------|---------------|
| $LOF \approx 1$ | $p$ has similar density to its neighbors → **normal** |
| $LOF \gg 1$ | $p$ is much less dense than its neighbors → **outlier** |

**Function summary:**

| Function | Role |
|----------|------|
| `fit(X)` | Stores training data. Precomputes the full pairwise distance matrix, the $k$-distance and the lrd of every training point. Doing this at fit time avoids redundant recomputation at prediction. |
| `_compute_pairwise_distances(X)` | Builds the $n \times n$ distance matrix using NumPy broadcasting — no Python loop over pairs. |
| `_k_neighbors_train(i)` | Returns the $k$ nearest neighbors of train point $i$ by reading row $i$ of the precomputed matrix. Skips index 0 (a train point's closest neighbor is itself, at distance 0). |
| `_k_neighbors_test(x)` | Returns the $k$ nearest neighbors of a new test point $x$ by computing distances to $X_{train}$ on the fly. Does **not** skip index 0 — $x$ is not in $X_{train}$, so its nearest neighbor is a genuine point. |
| `_reach_dist(dist_p_o, k_dist_o)` | Computes $\max(k\text{-distance}(o),\ d(p, o))$ for a single pair. |
| `_lrd_train(i)` | lrd of training point $i$ — reads all distances from the precomputed matrix. |
| `_lrd_test(x)` | lrd of a new test point $x$ — computes distances on the fly, but reads neighbor $k$-distances from the precomputed values stored at fit. |
| `score_samples(X)` | LOF score for each point in $X$ — ratio of mean neighbor lrd to point lrd. |
| `predict(X)` | Flags the top `contamination` fraction as outliers (`1`), rest as normal (`0`). |
| `fit_predict(X)` | Shorthand: calls `fit(X)` then `predict(X)` in one step. |

**Why precompute in `fit()`?** 
LOF needs the $k$-distances and lrd values of training points both when evaluating training points and when evaluating new test points. Precomputing them once at `fit()` time avoids recomputing the full distance matrix for every call to `predict()`.

---
## 3. Treatment Methods — Univariate

### What does "treating" an outlier mean?

Detection tells us *which* rows are outliers. Treatment answers the next question: **what do we do with them?**

The right answer depends on the context. Ask yourself:
- Is this outlier a **data error**? → Remove it or replace it with a representative value.
- Is it a **rare but real event**? → Keep it, but reduce its influence on the model.
- Do I **need to keep all rows**? → Never delete — replace the value instead.

**Univariate treatment** processes each column **independently**, one at a time. It is the natural complement to univariate detection (IQR, Z-Score).

All methods are implemented in the `Univariate_Treatment` class and accessible through a single dispatcher:

```python
treatment = Univariate_Treatment(df)
df_clean  = treatment.treat(method, outlier_indices=idx, **kwargs)
```



---

### 3.1 Removal

The simplest approach: **delete the row entirely**. The outlier value no longer exists in the dataset.

**When to use it:**
- The outlier is clearly a data entry error (age = 500, a negative price…)
- The dataset is large enough that losing a few rows has no significant impact
- You are certain the row carries no useful information

**When to avoid it:**
- Small dataset — every row matters
- The outlier is a genuine rare event (medical anomaly, fraud case)
- Systematic removal could bias your dataset toward common cases

```
Before: [22, 25, 27, 23, 24, 200]   ← age = 200, clearly a data error
After:  [22, 25, 27, 23, 24]         ← the entire row is dropped
```

| Function | Role |
|----------|------|
| `remove_outliers(outlier_indices)` | Calls `DataFrame.drop(index=outlier_indices)` on an internal copy of the data. The original dataframe passed at construction is never modified. |

---

### 3.2 Winsorizing (Capping)

Instead of deleting extreme values, we **clip** them to a boundary. The row is kept — only the value that is too extreme gets pulled back to an acceptable limit.

**Intuition:** imagine a company's salary dataset where one entry is \$50,000,000 due to a typo. Winsorizing would cap it at the 95th percentile — say \$180,000. The employee's record is preserved, the distortion is eliminated.

The lower bound $L$ and upper bound $U$ are computed from the data using quantiles:

$$L = Q_{\alpha}, \quad U = Q_{1-\alpha} \quad (\text{default: } \alpha = 0.05)$$

Each value is then replaced by:

$$x_i^* = \begin{cases} L & \text{if } x_i < L \\ U & \text{if } x_i > U \\ x_i & \text{otherwise} \end{cases}$$

Bounds are computed on **inlier rows only** — using the outliers themselves to compute bounds would contaminate the reference.

```
Column:      [10, 12, 11, 13, 200]
Q95 (inliers) = 13.9
After:       [10, 12, 11, 13, 13.9]   ← 200 is pulled back to the bound
```

| Function | Role |
|----------|------|
| `winsorize(outlier_indices, lower_quantile=0.05, upper_quantile=0.95)` | For each numeric column, computes Q5 and Q95 on inlier rows only, then applies `pandas.Series.clip(lower, upper)` to the full column — clipping brings any value outside the range back to the nearest bound. |

---

### 3.3 Imputation by Median

Replace the outlier value with the **median** of the normal (inlier) values in that column.

**Why the median and not the mean?** 
The median is the middle value when all values are sorted. It is **robust to extreme values**: adding one very large number barely moves the median, but it can shift the mean dramatically. For skewed distributions, the median is a better representative of the "typical" value.


$$x_i^* = \text{median}\left(\{x_j : j \notin \text{outlier\_indices}\}\right)$$

```
Column: [22, 25, 27, 23, 24, 200]
Inlier median = median([22, 25, 27, 23, 24]) = 24
After:  [22, 25, 27, 23, 24, 24]   ← 200 replaced by 24
```

| Function | Role |
|----------|------|
| `impute_median(outlier_indices)` | Builds a boolean inlier mask with `~index.isin(outlier_indices)`, computes `.median()` on the masked column, then replaces the outlier cell values with that median. Operates column by column. |

---

### 3.4 Imputation by Mean

Replace the outlier value with the **mean** of the inlier values.

**When to prefer the mean over the median?** 
When the distribution is approximately symmetric (Gaussian-like). In that case, mean and median are close to each other, and the mean is a slightly more precise estimate because it uses all values, not just the central one.

$$x_i^* = \bar{x}_{\text{inliers}} = \frac{1}{n_{\text{inliers}}} \sum_{j \notin \text{outliers}} x_j$$


| Function | Role |
|----------|------|
| `impute_mean(outlier_indices)` | Same logic as `impute_median`, but calls `.mean()` on the inlier rows. Also casts the column to `float` before writing, to safely accept the computed mean (which is always a float). |

---

### 3.5 Log Transform

Apply a logarithm to **compress the scale** of the entire column. Extreme values become much smaller relative to normal values — without deleting anything.

$$x_i^* = \log(1 + x_i)$$

We use $\log(1 + x)$ instead of $\log(x)$ for one specific reason: $\log(0) = -\infty$, while $\log(1 + 0) = 0$. This makes the transform safe for columns that contain zeros.

```
Before: [10,   100,   1 000,  10 000,  100 000]
After:  [2.4,  4.6,   6.9,    9.2,     11.5  ]
                                  ↑ scale compressed dramatically
```

This method is Only applicable to **non-negative columns**. Log is undefined for negative numbers.

| Function | Role |
|----------|------|
| `log_transform(columns=None)` | Applies `numpy.log1p()` (which computes $\log(1+x)$ numerically stably) to each target column. If a column contains any negative value, it is skipped with a warning. If `columns=None`, all numeric columns are transformed. |

---

### 3.6 Square Root Transform

A softer alternative to the log transform. Instead of a logarithm, we apply a square root — the compression is less aggressive.

$$x_i^* = \sqrt{x_i}$$

```
Before: [1,   4,   9,   100,   10 000]
After:  [1,   2,   3,   10,    100   ]   ← extreme values reduced, but gently
```



| Function | Role |
|----------|------|
| `sqrt_transform(columns=None)` | Applies `numpy.sqrt()` to each target column. Skips any column that contains negative values. If `columns=None`, all numeric columns are transformed. |

---



---
## 4. Treatment Methods — Multivariate

### Why a separate multivariate treatment?

Consider this example:

| height | weight | Is it an outlier? |
|--------|--------|-------------------|
| 175 cm | 70 kg  |  Normal — realistic combination |
| 150 cm | 120 kg |  Outlier — abnormal height/weight ratio |

Looking at each column alone: 150 cm is a normal height, 120 kg is an unusual but plausible weight. **Together**, the combination is anomalous — it falls far outside the typical relationship between height and weight.

Univariate treatment would miss this entirely, because it processes each column in isolation. **Multivariate treatment** considers **all features simultaneously** — both when identifying which rows are problematic, and when replacing their values.

All methods are implemented in `Multivariate_Treatment` and accessible through:

```python
treatment = Multivariate_Treatment(df)
df_clean  = treatment.treat(method, outlier_indices=idx, **kwargs)
```

| Method | Key idea |
|--------|----------|
| `remove` | Drop the entire row |
| `impute_median` | Replace all columns of the row with their inlier medians |
| `impute_knn` | Replace with a weighted average of the $k$ most similar normal rows |

---

### 4.1 Removal

Identical in mechanics to univariate removal — drop the entire row. The difference is **why** the row is being dropped: it was flagged because of an **abnormal combination** of feature values (by LOF or Isolation Forest), not because of a single extreme column.

| Function | Role |
|----------|------|
| `remove_outliers(outlier_indices)` | Calls `DataFrame.drop(index=outlier_indices)` on an internal copy. Returns the cleaned DataFrame. |

---

### 4.2 Median Imputation

For each outlier row, replace **all its numeric columns simultaneously** with their respective inlier column medians.

**Why replace all columns?** 
The row was flagged as a multivariate outlier because of the *combination* of its values. Fixing only one column would leave the row in an inconsistent state — the other columns would still be part of the anomalous combination. Replacing all columns at once ensures the row becomes representative.

$$x_{i,j}^* = \text{median}\left(\{x_{k,j} : k \notin \text{outliers}\}\right) \quad \text{for all columns } j$$

```
Outlier row:    height=150, weight=120, age=32
Inlier medians: height=172, weight=68,  age=29
After:          height=172, weight=68,  age=29   ← whole row replaced
```

| Function | Role |
|----------|------|
| `impute_median(outlier_indices)` | Builds an inlier boolean mask, computes the median per column on inlier rows, then replaces all numeric values of each outlier row with the corresponding column median. |

---

### 4.3 KNN Imputation — The Most Powerful Method

Instead of replacing an outlier row with a generic column median, KNN finds the $k$ most **similar normal rows** in the dataset and computes a **weighted average** of their values — closer neighbors contribute more than distant ones.





The algorithm proceeds in four steps for each outlier row:

**Step 1 — Euclidean distance to all inlier rows** 
We compute the distance from the outlier point to every normal point, using all numeric features simultaneously:

$$d(p, q) = \sqrt{\sum_{j=1}^{d}(p_j - q_j)^2}$$

**Step 2 — Select the $k$ nearest inliers** 
Sort all distances and keep the $k$ smallest — these are the most similar normal rows.

**Step 3 — Inverse-distance weights** 
Closer neighbors should have more influence. We weight each neighbor by the inverse of its distance:

$$w_i = \frac{1}{d(p, q_i) + \varepsilon}, \qquad \tilde{w}_i = \frac{w_i}{\sum_j w_j}$$

The small constant $\varepsilon = 10^{-10}$ prevents division by zero if two points are perfectly identical.

**Step 4 — Weighted imputation per feature** 
For each column $j$, the imputed value is the weighted average of the $k$ neighbors' values:

$$x_{p,j}^* = \sum_{i=1}^{k} \tilde{w}_i \cdot x_{q_i, j}$$

```
Outlier row: height=150, weight=120

3 nearest inlier neighbors:
  q1 (d=5.2):  height=168, weight=65  → w̃ = 0.35
  q2 (d=6.1):  height=171, weight=70  → w̃ = 0.29
  q3 (d=9.3):  height=165, weight=62  → w̃ = 0.19   (others share the rest)

Imputed height = 0.35×168 + 0.29×171 + 0.19×165 + … ≈ 168.4
```

| Function | Role |
|----------|------|
| `impute_knn(outlier_indices, n_neighbors=5)` | Main entry point. For each outlier row: separates inliers from outliers, calls `_euclidean_distances()` to get all distances, picks the $k$ nearest, computes inverse-distance weights, then imputes each numeric column with the weighted average of the $k$ neighbors. |
| `_euclidean_distances(point, matrix)` | **Helper function.** Computes the Euclidean distance from a single 1D outlier point to every row of the inlier matrix. Uses NumPy broadcasting: `matrix - point` subtracts the outlier vector from every inlier row in a single vectorized operation, avoiding a slow Python loop. Result is a 1D array of length `n_inliers`. |

---
## 5. Implementation

### Synthetic Data
We build a clean dataset first, then inject known outliers at fixed indices.
This lets us measure how well each method restores the original values.



In [1]:
import time
import numpy as np
import pandas as pd

np.random.seed(42)


N_NORMAL   = 500
N_OUTLIERS = 30
N_FEATURES = 4
OUTLIER_IDX = list(range(N_NORMAL, N_NORMAL + N_OUTLIERS))  # last 30 rows

# Clean data — Gaussian, realistic scale
X_clean = np.column_stack([
    np.random.normal(170, 10, N_NORMAL + N_OUTLIERS),   # height (cm)
    np.random.normal(70,   8, N_NORMAL + N_OUTLIERS),   # weight (kg)
    np.random.normal(35,   5, N_NORMAL + N_OUTLIERS),   # age
    np.random.normal(50000, 8000, N_NORMAL + N_OUTLIERS) # salary
])

# Save the "true" clean values for outlier rows (our ground truth for MSE)
X_ground_truth = X_clean[OUTLIER_IDX].copy()

# Inject outliers — extreme values at known positions
X_dirty = X_clean.copy()
X_dirty[OUTLIER_IDX, 0] = np.random.uniform(220, 260, N_OUTLIERS)    # height anomaly
X_dirty[OUTLIER_IDX, 1] = np.random.uniform(150, 200, N_OUTLIERS)    # weight anomaly
X_dirty[OUTLIER_IDX, 2] = np.random.uniform(120, 150, N_OUTLIERS)    # age anomaly
X_dirty[OUTLIER_IDX, 3] = np.random.uniform(500000, 900000, N_OUTLIERS) # salary anomaly

df_dirty = pd.DataFrame(X_dirty, columns=["height", "weight", "age", "salary"])

print(f"Dataset: {N_NORMAL} normal rows + {N_OUTLIERS} injected outliers")
print(f"Outlier indices: {OUTLIER_IDX[0]} → {OUTLIER_IDX[-1]}")
print(f"\nSample outlier row:\n{df_dirty.iloc[OUTLIER_IDX[0]]}")


def mse_on_outliers(df_treated, ground_truth, outlier_idx):
    """MSE between treated values and original clean values on outlier rows only."""
    treated_vals = df_treated.iloc[outlier_idx].values
    return round(float(np.mean((treated_vals - ground_truth) ** 2)), 2)

def mean_shift(df_original, df_treated):
    """Average absolute change in column means — measures global data distortion."""
    shift = np.abs(df_original.mean() - df_treated.mean()).mean()
    return round(float(shift), 4)

def run_treatment(name, fn, n_runs=5):
    """Runs fn() n_runs times, returns avg time + quality metrics."""
    times, result = [], None
    for _ in range(n_runs):
        t0 = time.perf_counter()
        result = fn()
        times.append(time.perf_counter() - t0)
    return {
        "method": name,
        "time_ms":    round(np.mean(times) * 1000, 2),
        "mse_outliers": mse_on_outliers(result, X_ground_truth, OUTLIER_IDX),
        "mean_shift":   mean_shift(df_dirty, result),
    }

print("\nHelpers ready.")

Dataset: 500 normal rows + 30 injected outliers
Outlier indices: 500 → 529

Sample outlier row:
height       241.237507
weight       194.928030
age          131.644971
salary    519991.611464
Name: 500, dtype: float64

Helpers ready.


<img src="../assets/imgs/outliers/benchmark2.png" width="1000">

## With Scikit-learn

In [15]:
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import FunctionTransformer


results = []

sk_median = SimpleImputer(strategy="median")
# sklearn imputes NaN — we temporarily mask outlier rows with NaN for fair comparison
def sklearn_median():
    df_nan = df_dirty.copy()
    df_nan.iloc[OUTLIER_IDX] = np.nan
    return pd.DataFrame(sk_median.fit_transform(df_nan), columns=df_dirty.columns)

results.append(run_treatment("[SKLEARN]  SimpleImputer median", sklearn_median))

#Mean imputation

sk_mean = SimpleImputer(strategy="mean")
def sklearn_mean():
    df_nan = df_dirty.copy()
    df_nan.iloc[OUTLIER_IDX] = np.nan
    return pd.DataFrame(sk_mean.fit_transform(df_nan), columns=df_dirty.columns)

results.append(run_treatment("[SKLEARN]  SimpleImputer mean", sklearn_mean))


#  Winsorizing

def sklearn_winsorize():
    df_w = df_dirty.copy()
    inlier_mask = ~df_w.index.isin(OUTLIER_IDX)  
    for col in df_w.columns:
        lo = df_w.loc[inlier_mask, col].quantile(0.05)  
        hi = df_w.loc[inlier_mask, col].quantile(0.95)  
        df_w[col] = df_w[col].clip(lo, hi)
    return df_w

results.append(run_treatment("[NUMPY]    clip (winsorize equiv)", sklearn_winsorize))

#knn
sk_knn = KNNImputer(n_neighbors=5)
def sklearn_knn():
    df_nan = df_dirty.copy()
    df_nan.iloc[OUTLIER_IDX] = np.nan
    return pd.DataFrame(sk_knn.fit_transform(df_nan), columns=df_dirty.columns)

results.append(run_treatment("[SKLEARN]  KNNImputer", sklearn_knn))


#Log transform

def sklearn_log():
    ft = FunctionTransformer(np.log1p)
    return pd.DataFrame(ft.fit_transform(df_dirty), columns=df_dirty.columns)

log_times_ours, log_times_sk = [], []
for _ in range(5):
    t0 = time.perf_counter(); sklearn_log(); log_times_sk.append(time.perf_counter()-t0)
    
results.append({"method": "[SKLEARN]  FunctionTransformer log",
                "time_ms": round(np.mean(log_times_sk)*1000,2),
                "mse_outliers": "N/A (scale change)", "mean_shift": "N/A"})

## With ifri_mini_ml_lib

In [ ]:
from ifri_mini_ml_lib.preprocessing.outlier_management.Treatment.univariate_treatment import Univariate_Treatment
from ifri_mini_ml_lib.preprocessing.outlier_management.Treatment.multivariate_treatment import Multivariate_Treatment


results.append(run_treatment(
    "[OURS]    Univariate median",
    lambda: Univariate_Treatment(df_dirty).treat("impute_median", outlier_indices=OUTLIER_IDX)
))


# ── Mean imputation 
results.append(run_treatment( 
    "[OURS]    Univariate mean",
    lambda: Univariate_Treatment(df_dirty).treat("impute_mean", outlier_indices=OUTLIER_IDX)
))


results.append(run_treatment(
    "[OURS]    Univariate winsorize",
    lambda: Univariate_Treatment(df_dirty).treat(
        "winsorize",
        outlier_indices=OUTLIER_IDX
    )
))


# KNN imputation
results.append(run_treatment(
    "[OURS]    Multivariate KNN",
    lambda: Multivariate_Treatment(df_dirty).treat("impute_knn", outlier_indices=OUTLIER_IDX, n_neighbors=5)
))

#Log transform 

def ours_log():
    return Univariate_Treatment(df_dirty).treat("log")
log_times_ours, log_times_sk = [], []

for _ in range(5):
    t0 = time.perf_counter(); ours_log(); log_times_ours.append(time.perf_counter()-t0)
    
df_bench = pd.DataFrame(results)
print(df_bench.to_string(index=False))

## ifri_mini_ml_lib vs Sckit-learn 

<img src="../assets/imgs/outliers/benchmark1.png" width="900">

## 6. Real-Life Applications

Outlier processing is not just a mathematical sanitization process; it is vital across distinct core domains:

1.  **Financial Fraud Systems:** In banking ecosystem databases, transactions with highly skewed properties are captured via multivariate models. Legitimate high-net-worth variations must be swiftly distinguished from actual structural fraud patterns before final feature scaling.
2.  **Healthcare & Medical Devices:** In ICU patient vitals streams (ECG, blood pressure), a single isolated outlier can indicate a physical sensor disconnection (requiring univariant imputation) or a catastrophic cardiovascular event (requiring distinct categorical warning isolation).
3.  **Industrial IoT Failure Prediction:** Factory heavy machinery telemetry fields track simultaneous heat and acoustic vibration levels. When an asset displays anomalous internal physical correlation coordinates, automated imputation structures help identify whether structural deterioration is underway.

## 7. Limitations and Challenges of Outlier Processing

Just like detection, algorithmic processing of outliers presents challenges:

1. **The Curse of Dimensionality:** In distance-based algorithms (such as LOF or multivariate KNN imputation), the more columns there are, the more the distances between points equalize. It then becomes mathematically difficult to distinguish the "neighbor" from the "anomaly."

2. **Information Loss vs. Noise:** The `remove` method is drastic and can destroy valuable information if the dataset is small. Conversely, imputation can create artificial data that reduces the natural variance of the phenomenon being studied.

3. **The critical choice of $K$:** In KNN imputation or LOF, choosing a $K$ that is too small makes the algorithm hypersensitive to local noise, while a $K$ that is too large dilutes the anomaly within the overall data set.



## 8. References
* Rousseeuw, P. J., & Hubert, M. (2011). *Robust statistics for outlier detection*. Wiley Interdisciplinary Reviews: Data Mining and Knowledge Discovery.
* Breunig, M. M., Kriegel, H. P., Ng, R. T., & Sander, J. (2000). *LOF: identifying density-based local outliers*. In ACM sigmod record.
* Scikit-Learn Documentation: *Imputation of missing values under distance parameters*. https://scikit-learn.org/stable/modules/impute.html
* Youtube :LecoinStat : https://youtu.be/gw1IJ7s2lu0?si=ahnI1f5ai8oDYA0N
* Geeksforgeeks : https://www.geeksforgeeks.org/data-analysis/univariate-bivariate-and-multivariate-data-and-its-analysis/

* Medium : https://medium.com/@samiraalipour/a-comprehensive-guide-to-outliers-in-machine-learning-detection-handling-and-impact-f7d965bba7a5